In [9]:
# Install deps (Colab VM)
# %%capture
# !apt-get update -y
# !apt-get install -y ffmpeg
# !pip -q install -U yt-dlp faster-whisper
# !pip -q install -U llama-cpp-python
# !pip -q install python-dotenv

In [10]:
# Install deps (Mac)
# brew update
# brew install ffmpeg
# pip install -U yt-dlp faster-whisper
# CMAKE_ARGS="-DLLAMA_METAL=on" pip install -U llama-cpp-python
# xcode-select --install
# pip install -U pip setuptools wheel
# pip install torch
# pip install python-dotenv

In [11]:
# Setup the environment or use Anaconda
# python -m venv venv
# source venv/bin/activate

In [12]:
# Mount Google Drive (persistent storage)
# from pathlib import Path
# drive.mount("/content/drive")

In [13]:
# Configure job + paste link
import re, json, subprocess, math
from pathlib import Path

URL = "https://www.youtube.com/watch?v=O1Z5lxyzR-o"

VIDEO_TITLE = ("5 Years ILR Call For Migrants In The Uk | New ILR Debate Outcome")

# Choose a stable job folder name (use anything you like)
# JOB_NAME = "lecture_job_01"
JOB_NAME = "migrants_in_the_uk"

# BASE = Path("/content/drive/MyDrive/transcriptions") / JOB_NAME
BASE = Path("output") / JOB_NAME

BASE.mkdir(parents=True, exist_ok=True)

WORK = BASE / "work"
WORK.mkdir(parents=True, exist_ok=True)

CHUNKS_DIR = BASE / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_RESULTS = BASE / "chunk_results"
CHUNK_RESULTS.mkdir(parents=True, exist_ok=True)

FINAL_DIR = BASE / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

BASE

PosixPath('output/migrants_in_the_uk')

In [14]:
# Download video (resumable-ish)
outtmpl = str(WORK / "%(title).200s.%(ext)s")

subprocess.run(["yt-dlp", "-f", "bv*+ba/best/best", "-o", outtmpl, URL], check=True)

video = max(WORK.glob("*"), key=lambda p: p.stat().st_mtime)
video

[youtube] Extracting URL: https://www.youtube.com/watch?v=O1Z5lxyzR-o
[youtube] O1Z5lxyzR-o: Downloading webpage


[youtube] O1Z5lxyzR-o: Downloading android vr player API JSON
[info] O1Z5lxyzR-o: Downloading 1 format(s): 399+251
[download] Destination: output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.f399.mp4
[download] 100% of   79.39MiB in 00:01:11 at 1.11MiB/s     
[download] Destination: output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.f251.webm
[download] 100% of   13.25MiB in 00:00:13 at 986.81KiB/s 
[Merger] Merging formats into "output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.webm"
Deleting original file output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.f251.webm (pass -k to keep)
Deleting original file output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.f399.mp4 (pass -k to keep)


PosixPath('output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.webm')

In [15]:
# Extract normalized WAV (16k mono)
audio = WORK / "audio.wav"

subprocess.run([
    "ffmpeg", "-y",
    "-i", str(video),
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    str(audio)
], check=True)

audio

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_2 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, matroska,webm, from 'output/migrants_in_the_uk/work/5 Years ILR Call For Migrants In The Uk ｜ New ILR Debate Outcome.webm':
  Metadata:
    COMPATIBLE_BRANDS: iso6av01mp41
    MAJOR_BRAND     : dash
    MINOR_

PosixPath('output/migrants_in_the_uk/work/audio.wav')

In [16]:
# Chunk audio into N-minute segments (recommended for long lectures)
CHUNK_SECONDS = 20 * 60  # 20 minutes

# Create chunks as WAV for easy timestamp handling
# -reset_timestamps 1 makes each chunk start at 0 internally
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        str(audio),
        "-f",
        "segment",
        "-segment_time",
        str(CHUNK_SECONDS),
        "-reset_timestamps",
        "1",
        "-c",
        "copy",
        str(CHUNKS_DIR / "chunk_%05d.wav"),
    ],
    check=True,
)

chunks = sorted(CHUNKS_DIR.glob("chunk_*.wav"))
len(chunks), chunks[:3]

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_2 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
[aist#0:0/pcm_s16le @ 0x131e105b0] Guessed Channel Layout: mono
Input #0, wav, from 'output/migrants_in_the_uk/work/audio.wav':
  Metadata:
    encoder         : Lavf62.3.100
  Duration: 00:14:25.88, bitrate: 256 kb/s
 

(1, [PosixPath('output/migrants_in_the_uk/chunks/chunk_00000.wav')])

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
print(hf_token[:8], "...")
os.environ["HF_TOKEN"] = hf_token

hf_pzeVU ...


In [18]:
# Transcribe with resume support (skips chunks already done)
from faster_whisper import WhisperModel
import torch

MODEL = "small"  # try: "base", "small", "medium", "large-v3"
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print("Device:", device, "compute:", compute_type)

model = WhisperModel(MODEL, device=device, compute_type=compute_type)


def transcribe_chunk(chunk_path: Path, out_json: Path, out_txt: Path):
    segments, info = model.transcribe(
        str(chunk_path),
        vad_filter=True,
        beam_size=5,
    )

    segs = []
    lines = []
    for s in segments:
        segs.append(
            {"start": float(s.start), "end": float(s.end), "text": s.text.strip()}
        )
        lines.append(s.text.strip())

    out_json.write_text(
        json.dumps(
            {
                "language": info.language,
                "language_probability": info.language_probability,
                "segments": segs,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    out_txt.write_text("\n".join(lines), encoding="utf-8")


# Process all chunks; resume if outputs exist
done = 0
for i, ch in enumerate(chunks):
    out_dir = CHUNK_RESULTS / f"{i:05d}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_json = out_dir / "segments.json"
    out_txt = out_dir / "chunk.txt"

    if out_json.exists() and out_txt.exists():
        done += 1
        continue

    print(f"Transcribing chunk {i+1}/{len(chunks)}: {ch.name}")
    transcribe_chunk(ch, out_json, out_txt)

print("Already done / skipped:", done, "of", len(chunks))

Device: cpu compute: int8
Transcribing chunk 1/1: chunk_00000.wav
Already done / skipped: 0 of 1


In [19]:
# Merge all chunks into final TXT + SRT + VTT + JSON (with correct offsets)
def srt_ts(t):
    ms = int(round(t * 1000))
    h = ms // 3600000
    ms %= 3600000
    m = ms // 60000
    ms %= 60000
    s = ms // 1000
    ms %= 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def vtt_ts(t):
    ms = int(round(t * 1000))
    h = ms // 3600000
    ms %= 3600000
    m = ms // 60000
    ms %= 60000
    s = ms // 1000
    ms %= 1000
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


all_segments = []
all_text_lines = []

# Load chunk outputs and apply time offsets
for i in range(len(chunks)):
    out_dir = CHUNK_RESULTS / f"{i:05d}"
    seg_path = out_dir / "segments.json"
    txt_path = out_dir / "chunk.txt"

    if not seg_path.exists():
        raise FileNotFoundError(f"Missing chunk result: {seg_path}")

    data = json.loads(seg_path.read_text(encoding="utf-8"))
    segs = data["segments"]
    offset = i * CHUNK_SECONDS

    for s in segs:
        all_segments.append(
            {"start": s["start"] + offset, "end": s["end"] + offset, "text": s["text"]}
        )

    if txt_path.exists():
        all_text_lines.append(txt_path.read_text(encoding="utf-8").strip())

# Write final JSON
final_json = FINAL_DIR / "transcript.json"
final_json.write_text(
    json.dumps(all_segments, ensure_ascii=False, indent=2), encoding="utf-8"
)

# Write final TXT
final_txt = FINAL_DIR / "transcript.txt"
final_txt.write_text("\n\n".join([t for t in all_text_lines if t]), encoding="utf-8")

# Write final SRT
final_srt = FINAL_DIR / "transcript.srt"
srt_lines = []
for idx, s in enumerate(all_segments, 1):
    srt_lines += [
        str(idx),
        f"{srt_ts(s['start'])} --> {srt_ts(s['end'])}",
        s["text"],
        "",
    ]
final_srt.write_text("\n".join(srt_lines), encoding="utf-8")

# Write final VTT
final_vtt = FINAL_DIR / "transcript.vtt"
vtt_lines = ["WEBVTT", ""]
for s in all_segments:
    vtt_lines += [f"{vtt_ts(s['start'])} --> {vtt_ts(s['end'])}", s["text"], ""]
final_vtt.write_text("\n".join(vtt_lines), encoding="utf-8")

(final_txt, final_srt, final_vtt, final_json)

(PosixPath('output/migrants_in_the_uk/final/transcript.txt'),
 PosixPath('output/migrants_in_the_uk/final/transcript.srt'),
 PosixPath('output/migrants_in_the_uk/final/transcript.vtt'),
 PosixPath('output/migrants_in_the_uk/final/transcript.json'))

In [20]:
# Downlaod model
# # Install aria2
# %%capture
# !apt-get update -y
# !apt-get install -y aria2

In [21]:
# Download to Drive (resumable)
# from pathlib import Path

# dst_dir = Path("/content/drive/MyDrive/models")
# dst_dir.mkdir(parents=True, exist_ok=True)

# out_path = dst_dir / "Qwen2.5-14B-Instruct-Q4_K_M.gguf"

# url = "https://huggingface.co/bartowski/Qwen2.5-14B-Instruct-GGUF/resolve/main/Qwen2.5-14B-Instruct-Q4_K_M.gguf"

# !aria2c -c -x 8 -s 8 -k 1M "{url}" -d "{dst_dir}" -o "{out_path.name}"
# print("Saved to:", out_path)

In [22]:
# --- installs (run once per runtime) ---
# !pip -q install -U llama-cpp-python

In [23]:
# Path to your generated transcript (from your earlier pipeline)
# TRANSCRIPT_PATH = f"/content/drive/MyDrive/ml_projects/description_generator/final/{JOB_NAME}/transcript.txt"
TRANSCRIPT_PATH = Path("output") / JOB_NAME / "final" / "transcript.txt"

# Model stored on Drive (GGUF)
# MODEL_PATH = "/content/drive/MyDrive/models/Qwen2.5-14B-Instruct-Q4_K_M.gguf"
MODEL_PATH = Path("models") / "Qwen2.5-14B-Instruct-Q4_K_M.gguf"

# Output path
# OUTPUT_MD_PATH = "/content/drive/MyDrive/ml_projects/description_generator/final/student_description.md"
OUTPUT_MD_PATH = Path("output") / JOB_NAME / "final" / "student_description.md"

# Chunking controls for long transcripts
CHUNK_MAX_CHARS = 12000  # reduce if you hit context limits
CHUNK_SUMMARY_MAX_TOKENS = 450
FINAL_MAX_TOKENS = 1100

In [24]:
# ========= [CELL] Load model =========
from pathlib import Path
import torch
from llama_cpp import Llama

MODEL_PATH = Path(MODEL_PATH)
assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"

USE_GPU = torch.cuda.is_available()

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=8192,  # try 4096 if you get memory errors
    n_threads=8,
    n_gpu_layers=35 if USE_GPU else 0,  # reduce (e.g., 20) if it errors on GPU
    verbose=False,
)

print("GPU available:", USE_GPU)
print("Loaded:", MODEL_PATH.name)

llama_context: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64 

GPU available: False
Loaded: Qwen2.5-14B-Instruct-Q4_K_M.gguf


In [25]:
# ========= [CELL] Load transcript =========
from pathlib import Path

tp = Path(TRANSCRIPT_PATH)
assert tp.exists(), f"Transcript not found: {tp}"

transcript_text = tp.read_text(encoding="utf-8").strip()
print("Transcript chars:", len(transcript_text))
print(transcript_text[:800])

Transcript chars: 14663
What's the outcome of the debate that was held on the 2nd of February 2026 regarding
ILR?
How did it go?
What did the NPC say?
Did this go in our favor?
These are more as some of the questions that a lot of people are asking.
So in today's video, we're going into the details, we'll be looking at the highlights
and just generally looking at what this could mean for everyone.
So stay tuned, let's just get straight into it.
My name is Tochi, you're welcome to my channel, please like this video, give
a thumbs up as you're watching, so YouTube can recommend it to more people that need to
see it.
So all my returning subscribers, you guys are wonderful, welcome back to today's video
and if you're yet to subscribe, why haven't you done that yet?
Why?
Please click on subscribe button right now, tha


In [26]:
# ========= [CELL] Prompt builders (NO quiz/questions) =========
import re


def _prompt(system: str, user: str) -> str:
    return f"SYSTEM: {system}\n\nUSER: {user}\n\nASSISTANT:\n"


def split_into_chunks(text: str, max_chars: int = 12000):
    """
    Chunk by paragraphs to keep coherence. Works well for long lecture transcripts.
    """
    text = re.sub(r"\r\n", "\n", text.strip())
    text = re.sub(r"\n{3,}", "\n\n", text)

    paras = text.split("\n\n")
    chunks, buf = [], ""

    for p in paras:
        p = p.strip()
        if not p:
            continue
        if len(buf) + len(p) + 2 <= max_chars:
            buf = (buf + "\n\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            buf = p

    if buf:
        chunks.append(buf)

    return chunks


# def summarize_chunk(
#     chunk_text: str, idx: int, total: int, subject: str, topic: str
# ) -> str:
#     """
#     First-pass "map" summary: extract factual notes per chunk in clear bullets.
#     Do NOT include quizzes, questions, exercises, or answers.
#     """
#     system = (
#         "You are an expert tutor. Convert lecture transcript excerpts into clear, factual study notes. "
#         "Do NOT include any quizzes, questions, exercises, or answers. "
#         "Organize ideas using concise bullet points, numbers, or letters for hierarchy if needed."
#     )

#     user = f"""
# Subject: {subject}
# Topic: {topic}
# Excerpt {idx}/{total}

# Rules:
# - Only include factual content from this excerpt; do not invent or add details.
# - Use concise bullet points.
# - Capture definitions, distinctions, steps, key examples, and listed items.
# - You may group related ideas using numbers (1, 2, 3), letters (a, b, c), or combinations (1a, 1b).
# - Do NOT include quizzes, questions, exercises, or answers.

# Excerpt:
# \"\"\"{chunk_text}\"\"\"
# """
#     out = llm(
#         _prompt(system, user),
#         max_tokens=CHUNK_SUMMARY_MAX_TOKENS,
#         temperature=0.2,
#         top_p=0.9,
#         repeat_penalty=1.07,
#         stop=["USER:", "SYSTEM:"],
#     )
#     return out["choices"][0]["text"].strip()



# def build_final_description_prompt(
#     all_notes: str, subject: str, topic: str
# ) -> tuple[str, str]:
#     """
#     Produce a polished, student-friendly description in plain text.
#     Ideas can be grouped using numbers, letters, or simple combinations.
#     """
#     system = (
#         "You are an expert academic writer and tutor. Create clear, organized learning materials for students. "
#         "Do NOT include quizzes, questions, exercises, or answers. "
#         "Provide the output in plain text with headings. Use numbers, letters, or simple combinations to group ideas. "
#         "Do not use any symbols like # or * for formatting."
#     )

#     user = f"""
# Create a well-structured learning description from the notes below.

# Context:
# Subject: {subject}
# Video topic: {topic}

# Hard rules:
# - Do NOT generate quizzes, questions, exercises, or answers.
# - Use simple teaching language.
# - Keep it factual: do not add facts not supported by the notes.
# - Prioritize clarity for students.
# - Organize ideas with headings and short paragraphs.
# - You may group ideas using numbers (1, 2, 3), letters (a, b, c), or simple combinations (1a, 1b) to show hierarchy.
# - Do not use markdown symbols like #, *, or any other special characters for formatting.

# Output structure:

# Overview
# (2–4 sentences summarizing the topic)

# Key Ideas Explained
# (short paragraphs; group related points using numbers, letters, or combinations)

# Important Terms
# (term: definition)

# What Students Should Remember
# (5–10 plain text bullets; use dashes or numbers)

# Notes:
# \"\"\"{all_notes}\"\"\"
# """
#     return system, user


# def generate_final_description(all_notes: str, subject: str, topic: str) -> str:
#     system, user = build_final_description_prompt(all_notes, subject, topic)
#     out = llm(
#         _prompt(system, user),
#         max_tokens=FINAL_MAX_TOKENS,
#         temperature=0.25,
#         top_p=0.9,
#         repeat_penalty=1.08,
#         stop=["USER:", "SYSTEM:"],
#     )
#     return out["choices"][0]["text"].strip()

In [27]:
# ========= [CELL] Video summarization for students =========
import re

def _prompt(system: str, user: str) -> str:
    return f"SYSTEM: {system}\n\nUSER: {user}\n\nASSISTANT:\n"

def split_into_chunks(text: str, max_chars: int = 12000):
    """
    Chunk text by paragraphs to preserve coherence.
    Suitable for long video transcripts.
    """
    text = re.sub(r"\r\n", "\n", text.strip())
    text = re.sub(r"\n{3,}", "\n\n", text)

    paras = text.split("\n\n")
    chunks, buf = [], ""

    for p in paras:
        p = p.strip()
        if not p:
            continue
        if len(buf) + len(p) + 2 <= max_chars:
            buf = (buf + "\n\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            buf = p

    if buf:
        chunks.append(buf)

    return chunks

def summarize_chunk(chunk_text: str, idx: int, total: int, video_title: str) -> str:
    """
    Summarize a transcript chunk into clear study notes.
    Capture main ideas, explanations, examples, analogies.
    Avoid inventing facts or adding unrelated content.
    """
    system = (
        "You are an expert tutor. Convert any video transcript excerpt into clear, factual study notes. "
        "Include main ideas, explanations, examples, and important distinctions. "
        "Do NOT create quizzes, questions, exercises, or answers. "
        "Organize ideas using concise bullets, numbers, letters, or simple combinations if needed."
    )

    user = f"""
Video title: {video_title}
Excerpt {idx}/{total}

Rules:
- Only include content present in this excerpt; do not invent or add facts.
- Capture main ideas, explanations, examples, or analogies.
- Use concise bullet points.
- You may group related ideas using numbers (1, 2, 3), letters (a, b, c), or combinations (1a, 1b).
- Do NOT include quizzes, questions, exercises, or answers.

Excerpt:
\"\"\"{chunk_text}\"\"\"
"""
    out = llm(
        _prompt(system, user),
        max_tokens=CHUNK_SUMMARY_MAX_TOKENS,
        temperature=0.2,
        top_p=0.9,
        repeat_penalty=1.07,
        stop=["USER:", "SYSTEM:"],
    )
    return out["choices"][0]["text"].strip()

def generate_final_description(all_notes: str, video_title: str) -> str:
    """
    Create a polished, student-friendly summary from all chunk notes.
    Includes Overview, Key Ideas, Important Terms, and Takeaways.
    """
    system = (
        "You are an expert academic writer and tutor. Create clear, organized learning materials for students. "
        "Include explanations, examples, and key ideas. "
        "Do NOT include quizzes, questions, exercises, or answers. "
        "Provide output in plain text with headings. "
        "Use numbers, letters, or simple combinations to group ideas; avoid markdown symbols."
    )

    user = f"""
Create a well-structured learning description from the notes below.

Video title: {video_title}

Hard rules:
- Do NOT generate quizzes, questions, exercises, or answers.
- Use simple teaching language.
- Keep it factual: do not add facts not present in the notes.
- Capture explanations, main ideas, examples, and analogies.
- Organize ideas with headings and short paragraphs.
- Group ideas using numbers (1, 2, 3), letters (a, b, c), or simple combinations (1a, 1b) for hierarchy.
- Do not use symbols like # or * for formatting.

Output structure:

Overview
(2–4 sentences summarizing the topic)

Key Ideas Explained
(brief paragraphs; include examples, analogies, or main points; group related points)

Important Terms
(term: definition)

What Students Should Remember
(5–10 concise bullets with the most essential points; avoid repeating all details)

Notes:
\"\"\"{all_notes}\"\"\"
"""
    out = llm(
        _prompt(system, user),
        max_tokens=FINAL_MAX_TOKENS,
        temperature=0.25,
        top_p=0.9,
        repeat_penalty=1.08,
        stop=["USER:", "SYSTEM:"],
    )
    return out["choices"][0]["text"].strip()


In [29]:
# ========= [CELL] Map-Reduce generation (resumable via Drive files) =========
from pathlib import Path
import json

# Where per-chunk notes are stored (so you can resume after disconnect)
NOTES_DIR = Path(Path(OUTPUT_MD_PATH).parent) / "notes_cache"
NOTES_DIR.mkdir(parents=True, exist_ok=True)

chunks = split_into_chunks(transcript_text, max_chars=CHUNK_MAX_CHARS)
print("Total chunks:", len(chunks))

chunk_notes = []

for i, ch in enumerate(chunks):
    cache_file = NOTES_DIR / f"chunk_notes_{i:05d}.txt"

    if cache_file.exists():
        note = cache_file.read_text(encoding="utf-8").strip()
    else:
        print(f"Summarizing chunk {i+1}/{len(chunks)}...")
        note = summarize_chunk(ch, i + 1, len(chunks), VIDEO_TITLE)
        cache_file.write_text(note, encoding="utf-8")

    chunk_notes.append(note)

combined_notes = "\n\n".join([n for n in chunk_notes if n.strip()])

# Save combined notes as an intermediate artifact
combined_path = NOTES_DIR / "combined_notes.txt"
combined_path.write_text(combined_notes, encoding="utf-8")

print("Combined notes chars:", len(combined_notes))
print(combined_notes[:800])

Total chunks: 1
Summarizing chunk 1/1...
Combined notes chars: 2141
- **Outcome of the Debate on ILR (Indefinite Leave to Remain)**
  - Held on February 2nd, 2026.
  - MPs emphasized fairness and trust in their arguments.
  - Majority supported maintaining the five-year ILR period for existing migrants.

- **Key Points Raised:**
  1. **Impact on Healthcare Workers:**
     a. Many healthcare workers came with the expectation of ILR after five years.
     b. Extending to ten years would cause significant staff shortages.
     c. Royal College of Nursing reported that about 60% might leave if ILR period is extended.

  2. **Economic Concerns:**
     a. Changing rules retrospectively could harm the economy.
     b. Migrants contribute significantly through taxes and labor.
     c. Extending ILR to ten or fifteen years would discourage skilled workers from comi


In [30]:
# ========= [CELL] Final polished description + save to Drive =========
from pathlib import Path

final_description = generate_final_description(combined_notes, VIDEO_TITLE)
print(final_description[:2000])

out_path = Path(OUTPUT_MD_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(final_description, encoding="utf-8")

print("\nSaved description to:", out_path)

Overview
The video discusses the outcome of a debate on indefinite leave to remain (ILR) for migrants in the UK, held on February 2nd, 2026. The main focus was on whether to maintain or extend the five-year period required for ILR eligibility.

Key Ideas Explained

1. Impact on Healthcare Workers
   - Many healthcare workers came with the expectation of obtaining ILR after five years.
   - Extending this period to ten years could lead to significant staff shortages, as reported by the Royal College of Nursing, indicating that about 60% might leave if the rules change.

2. Economic Concerns
   - Changing the ILR rules retrospectively could harm the economy.
   - Migrants contribute significantly through taxes and labor; extending ILR periods would discourage skilled workers from coming to the UK.

3. Social Care Workers
   - Proposed changes could worsen staff shortages in social care, adding uncertainty for those already working under challenging conditions.

4. Family and Personal Cir